# 🌿 Plant Disease Detection - Google Colab Edition

**Trains on GPU** - 10-15x faster than CPU!

- 🥔 Potato — 3 classes
- 🍅 Tomato — 10 classes  
- 🫑 Bell Pepper — 2 classes

**Total: 15 classes, 20K+ images**

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted!")

## Step 2: Check GPU & Install Dependencies

In [ ]:
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if len(tf.config.list_physical_devices('GPU')) > 0:
    print(f"GPU Device: {tf.config.list_physical_devices('GPU')[0].name}")
    print("🔥 Training will be FAST!")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → Select GPU")

## Step 3: Import Libraries

In [ ]:
import os
import shutil
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight

np.random.seed(42)
tf.random.set_seed(42)
print("✓ Libraries imported")

## Step 4: Dataset Path (Change this to YOUR path!)

In [ ]:
# 🚨 IMPORTANT: Upload PlantVillage folder to your Google Drive first!
# Then update the path below:

DATASET_PATH = '/content/drive/MyDrive/PlantVillage'  # Change if needed

IMAGE_SIZE   = 256
BATCH_SIZE   = 32
EPOCHS       = 50
CHANNELS     = 3

SELECTED_PLANTS = ['Pepper__bell', 'Potato', 'Tomato']

print(f"Dataset path: {DATASET_PATH}")
print(f"Exists: {os.path.exists(DATASET_PATH)}")

if not os.path.exists(DATASET_PATH):
    print("\n❌ Dataset not found!")
    print("📁 Please upload PlantVillage folder to Google Drive")
    print("📝 Then update DATASET_PATH above")

## Step 5: Explore Dataset

In [ ]:
PLANT_KEYS = [
    ('Pepper__bell', 'Bell Pepper'),
    ('Potato',       'Potato'),
    ('Tomato',       'Tomato'),
]

def parse_folder_name(folder):
    for plant_key, plant_disp in PLANT_KEYS:
        if folder.startswith(plant_key):
            rest = folder[len(plant_key):].lstrip('_')
            disease = rest.replace('_', ' ').replace('  ', ' ').strip().title()
            return plant_key, plant_disp, disease or 'Unknown'
    return folder, folder.replace('_', ' ').title(), 'Unknown'

all_folders = sorted(os.listdir(DATASET_PATH))
selected_folders = [
    f for f in all_folders
    if os.path.isdir(os.path.join(DATASET_PATH, f))
    and any(f.startswith(p) for p in SELECTED_PLANTS)
]

print(f"Selected classes ({len(selected_folders)}):")
total_imgs = 0
prev_plant = None
for folder in selected_folders:
    plant_key, plant_disp, disease_disp = parse_folder_name(folder)
    if plant_key != prev_plant:
        print(f"\n  [{plant_disp}]")
        prev_plant = plant_key
    imgs = [f for f in os.listdir(os.path.join(DATASET_PATH, folder))
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"    {disease_disp:50s}: {len(imgs)} images")
    total_imgs += len(imgs)
print(f"\n  Total: {total_imgs} images")

## Step 6: Load Dataset (80/10/10 Split)

In [ ]:
def get_dataset_partitions(ds, train_split=0.8, val_split=0.1, shuffle_size=10000):
    ds_size = len(ds)
    ds = ds.shuffle(shuffle_size, seed=42)
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    return ds.take(train_size), ds.skip(train_size).take(val_size), ds.skip(train_size + val_size)

full_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    seed=42,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    label_mode='int'
)

CLASS_NAMES = full_dataset.class_names
NUM_CLASSES = len(CLASS_NAMES)

print(f"{NUM_CLASSES} classes detected:")
for i, c in enumerate(CLASS_NAMES):
    _, plant_disp, disease_disp = parse_folder_name(c)
    print(f"  {i:2d}: {plant_disp:12s} — {disease_disp}")

train_ds, val_ds, test_ds = get_dataset_partitions(full_dataset)
print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} batches")

## Step 7: Visualize Samples

In [ ]:
plt.figure(figsize=(16, 12))
for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        _, plant_disp, disease_disp = parse_folder_name(CLASS_NAMES[labels[i]])
        plt.title(f"{plant_disp}\n{disease_disp}", fontsize=8)
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8: Class Weights

In [ ]:
all_labels = []
for _, labels in train_ds:
    all_labels.extend(labels.numpy())
all_labels = np.array(all_labels)

counts = [int(np.sum(all_labels == i)) for i in range(NUM_CLASSES)]
short_labels = []
for c in CLASS_NAMES:
    _, pd, dd = parse_folder_name(c)
    short_labels.append(f"{pd[:3]}. {dd[:14]}")

plt.figure(figsize=(16, 5))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))
plt.bar(range(NUM_CLASSES), counts, color=colors)
plt.xticks(range(NUM_CLASSES), short_labels, rotation=45, ha='right', fontsize=8)
plt.title('Class Distribution', fontsize=13)
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

cw_array = class_weight.compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weights = {i: w for i, w in enumerate(cw_array)}
print("✓ Class weights computed")

## Step 9: Data Augmentation

In [ ]:
resize_and_rescale = keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1.0 / 255),
], name='resize_rescale')

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)
print("✓ Preprocessing pipeline ready")

## Step 10: Build CNN Model

In [ ]:
model = keras.Sequential([
    resize_and_rescale,
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS)),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(), layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'), layers.Dropout(0.4),
    layers.Dense(64, activation='relu'), layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax'),
], name='plant_disease_cnn')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
model.build((None, IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
model.summary()

## Step 11: Train Model 🚀

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('best_plant_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

print("\n🔥 Starting training on GPU...\n")

history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights
)

print("\n✅ Training complete!")

## Step 12: Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train', color='#3498db')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='#e74c3c')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'], label='Train', color='#3498db')
axes[1].plot(history.history['val_loss'], label='Validation', color='#e74c3c')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Best val accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

## Step 13: Evaluate on Test Set

In [ ]:
scores = model.evaluate(test_ds)
print(f"\nTest Loss: {scores[0]:.4f}")
print(f"Test Accuracy: {scores[1]*100:.2f}%")

## Step 14: Confusion Matrix

In [ ]:
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())
y_true = np.array(y_true); y_pred = np.array(y_pred)

short_names = [f"{pd[:3]}. {dd[:15]}" for _, pd, dd in [parse_folder_name(c) for c in CLASS_NAMES]]

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=short_names, yticklabels=short_names)
plt.title('Confusion Matrix', fontsize=13)
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=short_names))

## Step 15: Save Model

In [ ]:
model.save('plant_model.h5')
print("✓ Saved: plant_model.h5")

with open('class_names.txt', 'w') as f:
    for cn in CLASS_NAMES: f.write(cn + '\n')
print("✓ Saved: class_names.txt")

# Copy to Google Drive
output_dir = '/content/drive/MyDrive/PlantDiseaseModel'
os.makedirs(output_dir, exist_ok=True)
shutil.copy('plant_model.h5', f'{output_dir}/plant_model.h5')
shutil.copy('class_names.txt', f'{output_dir}/class_names.txt')
print(f"\n✅ Model saved to Google Drive: {output_dir}")
print("\n📥 Download these files and put in your backend/models/ folder")

## 🎉 Done!

**Next Steps:**
1. Download `plant_model.h5` and `class_names.txt` from Google Drive
2. Copy them to your project's `backend/models/` folder
3. Start backend: `python backend/main.py`
4. Start frontend: `cd frontend && npm start`